In [ ]:
# Cell 1: Mount Drive and set workspace

import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_baseline_dir():
    # Find final_project/baseline in MyDrive or Shared drives.
    candidates = [
        "/content/drive/MyDrive/final_project/baseline",
        "/content/drive/MyDrive/final_project/baseline/",
    ]

    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project") and "baseline" in dirs:
                return os.path.abspath(os.path.join(root, "baseline"))

    raise FileNotFoundError("Could not find final_project/baseline in Drive.")

BASE_DIR = find_baseline_dir()
os.chdir(BASE_DIR)

print("BASE_DIR =", BASE_DIR)
print("CWD =", os.getcwd())

Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/final_project/baseline
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline


In [ ]:
# Cell 2: Install vLLM and API dependencies

import sys
import subprocess

# Install uv first.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U", "uv"
])

# Install general dependencies.
subprocess.check_call([
    "uv", "pip", "install", "--system", "-U",
    "openai>=1.55.0",
    "requests",
    "tqdm==4.66.2",
    "psutil",
    "jsonschema",
    "numpy",
    "accelerate",
    "safetensors",
    "huggingface_hub",
])

# Remove optional packages that can cause CUDA/import conflicts.
subprocess.call([
    "uv", "pip", "uninstall", "--system", "-y",
    "torchcodec", "torchvision", "torchaudio", "sentence-transformers"
])

# Gemma 4 needs recent Transformers support.
subprocess.check_call([
    "uv", "pip", "install", "--system", "-U",
    "transformers>=5.5.0"
])

# Recent vLLM build for CUDA 13 / Blackwell.
# If this fails in your runtime, use the fallback command below manually.
subprocess.check_call([
    "uv", "pip", "install", "--system", "-U",
    "vllm",
    "--torch-backend=cu130",
    "--extra-index-url", "https://wheels.vllm.ai/nightly/cu130",
])

# Fallback only if the cu130 line above fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

import importlib.metadata as md
import torch
import vllm
import transformers
import openai
import tqdm

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)
print("openai:", openai.__version__)
print("tqdm:", tqdm.__version__)

for pkg in ["torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
vLLM: 0.22.1rc1.dev88+gafcb58071
transformers: 5.9.0
openai: 2.40.0
tqdm: 4.67.3
torchcodec: not installed
torchvision: 0.26.0+cu130
torchaudio: 2.11.0+cu130
sentence-transformers: not installed
Tue Jun  2 17:35:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+===========

In [ ]:
# Cell 3: Clone or update repository

import os
import subprocess

REPO_URL = "https://github.com/ali-mohmmadi/KGP-CuriousLLM.git"
REPO_DIR = "/content/KGP-CuriousLLM"

if not os.path.isdir(REPO_DIR):
    print("Cloning repository into:", REPO_DIR)
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("Repo already exists. Pulling latest changes...")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

print("Repo ready at:", REPO_DIR)
print("Repo root files:", os.listdir(REPO_DIR)[:15])

Cloning repository into: /content/KGP-CuriousLLM
Repo ready at: /content/KGP-CuriousLLM
Repo root files: ['images', 'kgp_main.py', '.git', 'create_dirs.py', 'T5_main.py', 'grid_search_mistral_main.py', 'quantize_mistral_main.py', 'requirements.txt', 'configs', 'MDR_main.py', 'kg_construct_main.py', 'README.md', '.gitignore', 'MDR_embedding_main.py', 'ft_mistral_main.py']


In [ ]:
# Cell 4: Add repo and baseline to Python path

import os
import sys

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("PYTHONPATH ready")
print("BASE_DIR in path:", BASE_DIR in sys.path)
print("REPO_DIR in path:", REPO_DIR in sys.path)

PYTHONPATH ready
BASE_DIR in path: True
REPO_DIR in path: True


In [ ]:
# Cell 5: Set Gemma 4, vLLM, dataset, and output paths

import os
from pathlib import Path

MODEL_NAME = "google/gemma-4-26B-A4B-it"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Large context budget for Gemma 4 on a 95 GB Blackwell GPU.
MAX_MODEL_LEN = 131072

# Keep some VRAM margin.
GPU_MEMORY_UTILIZATION = 0.92

MAX_NUM_SEQS = 1
MAX_NUM_BATCHED_TOKENS = 32768

# Thinking/output budget.
ANSWER_MAX_TOKENS = 16384

SERVER_LOG_PATH = Path("/content/vllm_gemma4_answer_server.log")
SERVER_PID_PATH = Path("/content/vllm_gemma4_answer_server.pid")

# HotpotQA paths.
HOTPOT_DATA_PATH = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "evidence",
    "hotpot_evidence_1000",
    "qwen_agent",
    "evidence.json",
)

HOTPOT_SAVE_DIR = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "answers",
    "hotpot_answers_gemma4",
)

HOTPOT_SAVE_PATH = os.path.join(
    HOTPOT_SAVE_DIR,
    "qwen_agent_responses.json",
)

# 2WikiMQA paths.
TWO_WIKI_DATA_PATH = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "evidence",
    "2wikimultihopqa_evidence_1000",
    "qwen_agent",
    "evidence.json",
)

TWO_WIKI_SAVE_DIR = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "answers",
    "wiki_answers_gemma4",
)

TWO_WIKI_SAVE_PATH = os.path.join(
    TWO_WIKI_SAVE_DIR,
    "qwen_agent_responses.json",
)

os.makedirs(HOTPOT_SAVE_DIR, exist_ok=True)
os.makedirs(TWO_WIKI_SAVE_DIR, exist_ok=True)

print("MODEL_NAME =", MODEL_NAME)
print("BASE_URL =", BASE_URL)
print("MAX_MODEL_LEN =", MAX_MODEL_LEN)
print("ANSWER_MAX_TOKENS =", ANSWER_MAX_TOKENS)

print("\nHOTPOT_DATA_PATH =", HOTPOT_DATA_PATH)
print("HOTPOT_SAVE_PATH =", HOTPOT_SAVE_PATH)

print("\nTWO_WIKI_DATA_PATH =", TWO_WIKI_DATA_PATH)
print("TWO_WIKI_SAVE_PATH =", TWO_WIKI_SAVE_PATH)

assert os.path.isfile(HOTPOT_DATA_PATH), f"Missing HotpotQA evidence file: {HOTPOT_DATA_PATH}"
assert os.path.isfile(TWO_WIKI_DATA_PATH), f"Missing 2WikiMQA evidence file: {TWO_WIKI_DATA_PATH}"

print("\nBoth evidence files exist.")

MODEL_NAME = google/gemma-4-26B-A4B-it
BASE_URL = http://localhost:8000/v1
MAX_MODEL_LEN = 131072
ANSWER_MAX_TOKENS = 16384

HOTPOT_DATA_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/evidence.json
HOTPOT_SAVE_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gemma4/qwen_agent_responses.json

TWO_WIKI_DATA_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/evidence/2wikimultihopqa_evidence_1000/qwen_agent/evidence.json
TWO_WIKI_SAVE_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_gemma4/qwen_agent_responses.json

Both evidence files exist.


In [ ]:
# Cell 6: Optional Hugging Face token setup

import os

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets.")
except Exception as e:
    print("Could not read Colab secrets:", repr(e))

print("HF_TOKEN available:", bool(os.environ.get("HF_TOKEN")))

Could not read Colab secrets: SecretNotFoundError('Secret HF_TOKEN does not exist.')
HF_TOKEN available: False


In [ ]:
# Cell 7: Start Gemma 4 vLLM server with thinking enabled

import os
import gc
import time
import shlex
import psutil
import subprocess
import requests
import torch
from pathlib import Path

def kill_process_tree(pid):
    # Kill a process and all children.
    try:
        parent = psutil.Process(int(pid))

        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass

        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

def find_or_download_gemma4_chat_template():
    # Find or download the Gemma 4 vLLM chat template.
    template_name = "tool_chat_template_gemma4.jinja"
    local_template = Path("/content") / template_name

    if local_template.exists() and local_template.stat().st_size > 0:
        return local_template

    search_roots = []

    try:
        import vllm as vllm_pkg
        vllm_path = Path(vllm_pkg.__file__).resolve()
        search_roots.extend([
            vllm_path.parent,
            vllm_path.parent.parent,
        ])
    except Exception:
        pass

    search_roots.extend([
        Path("/usr/local/lib/python3.12/dist-packages"),
        Path("/usr/local/lib/python3.12/site-packages"),
        Path("/content"),
    ])

    for root in search_roots:
        if not root.exists():
            continue

        try:
            matches = sorted(root.rglob(template_name))
        except Exception:
            matches = []

        for match in matches:
            if match.exists() and match.stat().st_size > 0:
                print("Found Gemma 4 chat template:", match)
                return match

    # Fallback download.
    try:
        url = (
            "https://raw.githubusercontent.com/vllm-project/vllm/main/"
            "examples/tool_chat_template_gemma4.jinja"
        )
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        local_template.write_text(r.text, encoding="utf-8")

        if local_template.exists() and local_template.stat().st_size > 0:
            print("Downloaded Gemma 4 chat template:", local_template)
            return local_template

    except Exception as e:
        print("Warning: could not download Gemma 4 chat template:", repr(e))

    return None

# Stop old PID from this notebook.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

gemma4_chat_template_path = find_or_download_gemma4_chat_template()

cmd = [
    "vllm", "serve", MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only QA workload.
    "--language-model-only",
    "--limit-mm-per-prompt", '{"image": 0, "audio": 0}',

    # Gemma 4 thinking mode.
    "--reasoning-parser", "gemma4",
    "--tool-call-parser", "gemma4",
    "--enable-auto-tool-choice",
    "--default-chat-template-kwargs", '{"enable_thinking": true}',

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--enable-prefix-caching",
    "--generation-config", "vllm",
    "--dtype", "bfloat16",
    "--trust-remote-code",
]

if gemma4_chat_template_path is not None:
    cmd.extend(["--chat-template", str(gemma4_chat_template_path)])
else:
    print("Warning: running without explicit Gemma 4 chat template.")

server_env = os.environ.copy()

# Blackwell / CUDA 13 settings.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Downloaded Gemma 4 chat template: /content/tool_chat_template_gemma4.jinja
Command:
vllm serve google/gemma-4-26B-A4B-it --host 0.0.0.0 --port 8000 --max-model-len 131072 --gpu-memory-utilization 0.92 --language-model-only --limit-mm-per-prompt '{"image": 0, "audio": 0}' --reasoning-parser gemma4 --tool-call-parser gemma4 --enable-auto-tool-choice --default-chat-template-kwargs '{"enable_thinking": true}' --max-num-seqs 1 --max-num-batched-tokens 32768 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code --chat-template /content/tool_chat_template_gemma4.jinja

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server.
PID: 2741
Log: /content/vllm_gemma4_answer_server.log


In [ ]:
# Cell 8: Wait for vLLM server and create OpenAI-compatible client

import time
import requests
from pathlib import Path
from openai import OpenAI

def tail_log(path, n=80):
    # Read last log lines.
    path = Path(path)
    if not path.exists():
        return ""

    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

print("OpenAI-compatible client is ready.")

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(EngineCore pid=3281) INFO 06-02 17:36:30 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=3281) INFO 06-02 17:36:30 [parallel_state.py:1422] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:58355 backend=nccl
(EngineCore pid=3281) INFO 06-02 17:36:30 [parallel_state.py:1735] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0, EPLB rank N/A
(EngineCore pid=3281) INFO 06-02 17:36:31 [topk_topp_sampler.py:70] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0; using PyTorch-native sampler.
(EngineCore pid=3281) INFO 06-02 17:36:31 [gpu_model_runner.py:5089] Starting to load model google/gemma-4-26B-A4B-it...
(EngineCore pid=3281) INFO 06-02 17:36:31 [vllm.py:993] Asynchronous scheduling is enabled.
(Engine

In [ ]:
# Cell 9: Load and inspect HotpotQA evidence file

import json
from collections import Counter

def load_json(file_path: str):
    # Load JSON file.
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

hotpot_data = load_json(HOTPOT_DATA_PATH)

print("num_records =", len(hotpot_data))
print("type_counts =", Counter(r.get("type", "unknown") for r in hotpot_data))
print("first_keys =", list(hotpot_data[0].keys()))

print("\nFirst question:")
print(hotpot_data[0]["question"])

print("\nFirst answer:")
print(hotpot_data[0].get("answer", ""))

print("\nFirst evidence preview:")
for i, ev in enumerate(hotpot_data[0].get("evidence", [])[:3], start=1):
    print(f"{i}.", ev[:500])

num_records = 1000
type_counts = Counter({'bridge': 700, 'comparison': 300})
first_keys = ['type', 'question', 'evidence', 'answer', 'supports']

First question:
Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?

First answer:
yes

First evidence preview:
1. Title: Operation Dragoon. Evidence: Operation Dragoon also had political implications.
2. Title: Operation Dragoon. Evidence: Despite these successes, there was criticism of Dragoon by some Allied generals and contemporary commentators such as Bernard Montgomery, Arthur R. Wilson, and Chester Wilmot in the aftermath, mostly because of its geo-strategic implications.
3. Title: Operation Dragoon. Evidence: In the northeast the German problems loomed as large.


In [ ]:
# Cell 10: Define repo-faithful prompts

prompt = """
    Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

    - The answer should be directly relevant to the question.
    - Provide the answer in a clear, straightforward format.
    - Limit your answer to no more than 6 words, focusing on the essential information requested.
    - If the provided contexts do not contain enough information to answer the question, respond with "Information not available".
    - Do not include any additional tokens, explanations, or information beyond the direct answer.

    QUESTION: {question}
    CONTEXT: {context}
    ANSWER: [Your concise answer here or "Information not available" if the answer cannot be determined from the contexts.]

    """

none_prompt = """Given the following question, create a final answer in English to the question.
    QUESTION: {question}
    ANSWER: [Please provide only the answer and keep the answer less than 6 words.]
    """

print("Prompts ready.")

Prompts ready.


In [ ]:
# Cell 11: Define Gemma response cleaner

import re

def strip_thinking_artifacts(text: str) -> str:
    # Remove possible thinking artifacts.
    if text is None:
        return ""

    text = str(text)

    # Generic thinking blocks.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)

    # Gemma 4 channel-style thinking blocks, if leaked.
    text = re.sub(
        r"<\|channel\>thought.*?<channel\|>",
        "",
        text,
        flags=re.DOTALL,
    )

    # Extra Gemma-style markers, if leaked.
    text = text.replace("<|channel|>final<channel|>", "")
    text = text.replace("<|channel|>analysis<channel|>", "")
    text = text.replace("<|channel|>thought<channel|>", "")

    return text.strip()

def clean_gemma_response(text: str) -> str:
    # Clean final answer while preserving repo-style output.
    if text is None:
        return ""

    text = strip_thinking_artifacts(text).strip()

    # Remove common prefixes.
    for prefix in ["ANSWER:", "Answer:", "answer:", "Final answer:", "Final Answer:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    # Remove markdown code fences.
    text = re.sub(r"^```(?:text|json)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text.strip())

    # Keep first non-empty line if model adds extra explanation.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[0].strip()

    # Remove quotes around short answers.
    text = text.strip().strip('"').strip("'").strip()

    # Normalize common fallback variants.
    if text.lower() in {
        "not available",
        "information unavailable",
        "unknown",
        "cannot determine",
        "not enough information",
        "insufficient information",
    }:
        text = "Information not available"

    return text

print("Cleaner ready.")

Cleaner ready.


In [ ]:
# Cell 12: Define vLLM Gemma thinking generation

import time

LLM_SAMPLING_KWARGS = {
    "temperature": 0.7,
    "top_p": 0.8,
    "presence_penalty": 0.0,
}

LLM_EXTRA_BODY = {
    "top_k": 20,
    "min_p": 0.0,
    "repetition_penalty": 1.0,
    "chat_template_kwargs": {
        "enable_thinking": True,
    },
}

def get_reasoning_from_message(msg):
    # Read reasoning field if vLLM exposes it.
    for attr in ["reasoning_content", "reasoning"]:
        value = getattr(msg, attr, None)
        if value:
            return value
    return None

def gemma_chat_completion(input_prompt: str, max_tokens: int = ANSWER_MAX_TOKENS):
    # Call Gemma 4 through vLLM with thinking enabled.
    messages = [
        {"role": "user", "content": input_prompt},
    ]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=max_tokens,
        **LLM_SAMPLING_KWARGS,
        extra_body=LLM_EXTRA_BODY,
    )

    msg = response.choices[0].message

    # vLLM should separate reasoning from final content.
    content = msg.content or ""

    # Do not store reasoning in final output.
    reasoning = get_reasoning_from_message(msg)

    cleaned = clean_gemma_response(content)

    if not cleaned:
        cleaned = "Information not available"

    return cleaned

print("Local Gemma vLLM thinking generation function ready.")

Local Gemma vLLM thinking generation function ready.


In [ ]:
# Cell 13: Build one HotpotQA input prompt for smoke test

sample = hotpot_data[0]

sample_type = sample["type"]
sample_question = sample["question"]
sample_contexts = sample["evidence"]
sample_gt = sample["answer"]

if sample_contexts:
    sample_contexts_text = "\n".join(
        f"{i}: {c}" for i, c in enumerate(sample_contexts, start=1)
    )
    sample_input_prompt = prompt.format(
        question=sample_question,
        context=sample_contexts_text,
    )
else:
    sample_input_prompt = none_prompt.format(question=sample_question)

print("Sample type:", sample_type)
print("Sample question:", sample_question)
print("Sample gt:", sample_gt)
print("\nPrompt preview:")
print(sample_input_prompt[:3000])

Sample type: comparison
Sample question: Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?
Sample gt: yes

Prompt preview:

    Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

    - The answer should be directly relevant to the question.
    - Provide the answer in a clear, straightforward format.
    - Limit your answer to no more than 6 words, focusing on the essential information requested.
    - If the provided contexts do not contain enough information to answer the question, respond with "Information not available".
    - Do not include any additional tokens, explanations, or information beyond the direct answer.

    QUESTION: Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?
    CONTEXT: 1: Title: Operation Dragoon. Evidence: Operation Dragoon also had political implication

In [ ]:
# Cell 14: Smoke test Gemma on one HotpotQA sample

sample_response = gemma_chat_completion(
    input_prompt=sample_input_prompt,
    max_tokens=ANSWER_MAX_TOKENS,
)

print("Question:", sample_question)
print("GT:", sample_gt)
print("Gemma response:", sample_response)

Question: Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?
GT: yes
Gemma response: Information not available


In [ ]:
# Cell 15: Define resume-safe repo-style answer generation pipeline

import os
import json
import gc
import traceback
from tqdm.auto import tqdm

required_output_keys = {"type", "question", "gt", "response"}

def atomic_write_json(path, data):
    # Write JSON atomically.
    tmp_path = path + ".tmp"

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    os.replace(tmp_path, path)

def load_existing_responses(save_path):
    # Load existing repo-style responses.
    if not os.path.isfile(save_path):
        return []

    try:
        with open(save_path, "r", encoding="utf-8") as f:
            existing = json.load(f)

        if isinstance(existing, list):
            return existing

    except Exception as e:
        print("Could not load existing responses:", repr(e))

    return []

def build_input_prompt_from_record(record):
    # Build the same prompt as the Qwen notebook.
    question = record["question"]
    contexts = record.get("evidence", [])

    if contexts:
        contexts_text = "\n".join(
            f"{i}: {c}" for i, c in enumerate(contexts, start=1)
        )
        input_prompt = prompt.format(
            question=question,
            context=contexts_text,
        )
    else:
        input_prompt = none_prompt.format(question=question)

    return input_prompt

def make_output_record(record, response_text):
    # Keep exactly the repo-style schema.
    return {
        "type": record["type"],
        "question": record["question"],
        "gt": record["answer"],
        "response": response_text,
    }

def pipeline_resume(data, save_path, dataset_name="dataset"):
    # Resume-safe generation with repo-style output.
    existing = load_existing_responses(save_path)

    responses = []
    done_questions = set()

    for r in existing:
        if isinstance(r, dict) and required_output_keys.issubset(set(r.keys())):
            responses.append({
                "type": r["type"],
                "question": r["question"],
                "gt": r["gt"],
                "response": r["response"],
            })
            done_questions.add(r["question"])

    if responses:
        print(f"Resuming {dataset_name} from existing output.")
        print("Existing valid responses:", len(responses))
    else:
        print(f"No existing valid output found for {dataset_name}. Starting from scratch.")

    progress = tqdm(data, total=len(data), desc=f"Generating {dataset_name}", dynamic_ncols=True)

    for record in progress:
        question = record["question"]

        if question in done_questions:
            progress.set_postfix({
                "status": "skipped",
                "saved": len(responses),
            })
            continue

        try:
            input_prompt = build_input_prompt_from_record(record)

            resp = gemma_chat_completion(
                input_prompt=input_prompt,
                max_tokens=ANSWER_MAX_TOKENS,
            )

            output_record = make_output_record(record, resp)

        except Exception as e:
            print("\nError on question:", question)
            print("Error:", repr(e))
            print(traceback.format_exc())

            output_record = make_output_record(
                record,
                "Information not available",
            )

        responses.append(output_record)
        done_questions.add(question)

        # Save after each question.
        atomic_write_json(save_path, responses)

        # Light cleanup.
        if len(responses) % 25 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        progress.set_postfix({
            "status": "ok",
            "saved": len(responses),
        })

    atomic_write_json(save_path, responses)

    return responses

print("Resume-safe pipeline ready.")

Resume-safe pipeline ready.


In [ ]:
# Cell 16: Run answer generation for HotpotQA

hotpot_responses = pipeline_resume(
    data=hotpot_data,
    save_path=HOTPOT_SAVE_PATH,
    dataset_name="HotpotQA",
)

print("Finished HotpotQA.")
print("Saved to:", HOTPOT_SAVE_PATH)
print("Total responses:", len(hotpot_responses))

No existing valid output found for HotpotQA. Starting from scratch.


Generating HotpotQA:   0%|          | 0/1000 [00:00<?, ?it/s]

Finished HotpotQA.
Saved to: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gemma4/qwen_agent_responses.json
Total responses: 1000


In [ ]:
# Cell 17: Verify saved HotpotQA answers

import os
import json
from collections import Counter

assert os.path.isfile(HOTPOT_SAVE_PATH), f"Missing output file: {HOTPOT_SAVE_PATH}"

hotpot_saved = load_json(HOTPOT_SAVE_PATH)

print("HOTPOT_SAVE_PATH =", HOTPOT_SAVE_PATH)
print("num_saved =", len(hotpot_saved))
print("type_counts =", Counter(r.get("type", "unknown") for r in hotpot_saved))

print("\nFirst saved response:")
print(json.dumps(hotpot_saved[0], indent=2, ensure_ascii=False)[:3000])

HOTPOT_SAVE_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gemma4/qwen_agent_responses.json
num_saved = 1000
type_counts = Counter({'bridge': 700, 'comparison': 300})

First saved response:
{
  "type": "comparison",
  "question": "Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?",
  "gt": "yes",
  "response": "Information not available"
}


In [ ]:
# Cell 18: Load and inspect 2WikiMQA evidence

import json
from collections import Counter

two_wiki_data = load_json(TWO_WIKI_DATA_PATH)

print("num_records =", len(two_wiki_data))
print("type_counts =", Counter(r.get("type", "unknown") for r in two_wiki_data))
print("first_keys =", list(two_wiki_data[0].keys()))

print("\nFirst question:")
print(two_wiki_data[0]["question"])

print("\nFirst answer:")
print(two_wiki_data[0].get("answer", ""))

print("\nFirst evidence preview:")
for i, ev in enumerate(two_wiki_data[0].get("evidence", [])[:5], start=1):
    print(f"{i}.", ev[:500])

num_records = 1000
type_counts = Counter({'bridge_comparison': 250, 'inference': 250, 'comparison': 250, 'compositional': 250})
first_keys = ['type', 'question', 'evidence', 'answer', 'supports']

First question:
Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?

First answer:
yes

First evidence preview:
1. Title: Pauline Auzou. Evidence: Taylor & Francis; January 1997. . p. 199.
2. Title: Martial Law (1991 film). Evidence: The film has yet to arrive onto DVD in the United States.
3. Title: Pauline Auzou. Evidence: Berg; 6 April 1995. . p. 34.
4. Title: Jean Rollin. Evidence: Le temps d'un visage (1990), Jean Rollin. Éd.
5. Title: Até que a Sorte nos Separe. Evidence: Até que a Sorte nos Separe (English: Till Luck Do Us Part) is a 2012 Brazilian comedy film directed by Roberto Santucci and starring Leandro Hassum and Danielle Winits.


In [ ]:
# Cell 19: Smoke test Gemma on one 2WikiMQA sample

two_wiki_sample = two_wiki_data[0]

two_wiki_sample_type = two_wiki_sample["type"]
two_wiki_sample_question = two_wiki_sample["question"]
two_wiki_sample_contexts = two_wiki_sample["evidence"]
two_wiki_sample_gt = two_wiki_sample["answer"]

if two_wiki_sample_contexts:
    two_wiki_sample_contexts_text = "\n".join(
        f"{i}: {c}" for i, c in enumerate(two_wiki_sample_contexts, start=1)
    )
    two_wiki_sample_input_prompt = prompt.format(
        question=two_wiki_sample_question,
        context=two_wiki_sample_contexts_text,
    )
else:
    two_wiki_sample_input_prompt = none_prompt.format(
        question=two_wiki_sample_question
    )

two_wiki_sample_response = gemma_chat_completion(
    input_prompt=two_wiki_sample_input_prompt,
    max_tokens=ANSWER_MAX_TOKENS,
)

print("Sample type:", two_wiki_sample_type)
print("Question:", two_wiki_sample_question)
print("GT:", two_wiki_sample_gt)
print("Gemma response:", two_wiki_sample_response)

Sample type: bridge_comparison
Question: Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?
GT: yes
Gemma response: Information not available


In [20]:
# Cell 20: Run answer generation for 2WikiMQA

two_wiki_responses = pipeline_resume(
    data=two_wiki_data,
    save_path=TWO_WIKI_SAVE_PATH,
    dataset_name="2WikiMQA",
)

print("Finished 2WikiMQA.")
print("Saved to:", TWO_WIKI_SAVE_PATH)
print("Total responses:", len(two_wiki_responses))

No existing valid output found for 2WikiMQA. Starting from scratch.


Generating 2WikiMQA:   0%|          | 0/1000 [00:00<?, ?it/s]

Finished 2WikiMQA.
Saved to: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_gemma4/qwen_agent_responses.json
Total responses: 1000


In [21]:
# Cell 21: Verify saved 2WikiMQA answers

import os
import json
from collections import Counter

assert os.path.isfile(TWO_WIKI_SAVE_PATH), f"Missing output file: {TWO_WIKI_SAVE_PATH}"

two_wiki_saved = load_json(TWO_WIKI_SAVE_PATH)

print("TWO_WIKI_SAVE_PATH =", TWO_WIKI_SAVE_PATH)
print("num_saved =", len(two_wiki_saved))
print("type_counts =", Counter(r.get("type", "unknown") for r in two_wiki_saved))

print("\nFirst saved response:")
print(json.dumps(two_wiki_saved[0], indent=2, ensure_ascii=False)[:3000])

TWO_WIKI_SAVE_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_gemma4/qwen_agent_responses.json
num_saved = 1000
type_counts = Counter({'bridge_comparison': 250, 'inference': 250, 'comparison': 250, 'compositional': 250})

First saved response:
{
  "type": "bridge_comparison",
  "question": "Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?",
  "gt": "yes",
  "response": "Information not available"
}


In [22]:
# Cell 22: Check output schema consistency

required_keys = {"type", "question", "gt", "response"}

def check_schema(name, saved_records):
    # Check repo-style output schema.
    bad_records = []

    for i, record in enumerate(saved_records):
        if set(record.keys()) != required_keys:
            bad_records.append((i, list(record.keys())))

    print("=" * 80)
    print(name)
    print("Expected keys:", required_keys)
    print("Bad records:", len(bad_records))

    if bad_records:
        print("First bad record:", bad_records[0])
    else:
        print("All records have the expected repo-style schema.")

check_schema("HotpotQA", hotpot_saved)
check_schema("2WikiMQA", two_wiki_saved)

HotpotQA
Expected keys: {'type', 'gt', 'response', 'question'}
Bad records: 0
All records have the expected repo-style schema.
2WikiMQA
Expected keys: {'type', 'gt', 'response', 'question'}
Bad records: 0
All records have the expected repo-style schema.


In [23]:
# Cell 23: Final path summary

print("HotpotQA answers:")
print(HOTPOT_SAVE_PATH)

print("\n2WikiMQA answers:")
print(TWO_WIKI_SAVE_PATH)

print("\nFiles exist:")
print("HotpotQA:", os.path.isfile(HOTPOT_SAVE_PATH))
print("2WikiMQA:", os.path.isfile(TWO_WIKI_SAVE_PATH))

print("\nRecord counts:")
print("HotpotQA:", len(load_json(HOTPOT_SAVE_PATH)) if os.path.isfile(HOTPOT_SAVE_PATH) else 0)
print("2WikiMQA:", len(load_json(TWO_WIKI_SAVE_PATH)) if os.path.isfile(TWO_WIKI_SAVE_PATH) else 0)

HotpotQA answers:
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gemma4/qwen_agent_responses.json

2WikiMQA answers:
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_gemma4/qwen_agent_responses.json

Files exist:
HotpotQA: True
2WikiMQA: True

Record counts:
HotpotQA: 1000
2WikiMQA: 1000
